In [ ]:
# membandit setup
import os
import sys
from pathlib import Path

# Ensure cleaned_repo (parent of membandit/) is on sys.path when running from notebooks/
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# Optional: set MEMORYBANDIT_DATA_DIR to your data root (defaults to /Volumes/T7 Shield)
# os.environ.setdefault("MEMORYBANDIT_DATA_DIR", "/path/to/data/root")

from membandit.plotting_style import set_plot_style
set_plot_style()


# Cross-Region Multivariate Gaussian-Copula PAC Analysis

Trial-level cross-region multivariate PAC (ROI-hemi group-to-group) for every trial; behavioral metadata (hit/miss, rpe_cat, etc.) is attached for post-hoc condition splits.


In [ ]:
"""
Import required libraries for multivariate GC-PAC analysis
"""
import os
import sys
import warnings
import inspect

# Force local patched tensorpac before any tensorpac imports
TP_LOCAL_ROOT = '/Users/salmanqasim/Documents/GitRepos/tensorpac'
if TP_LOCAL_ROOT in sys.path:
    sys.path.remove(TP_LOCAL_ROOT)
sys.path.insert(0, TP_LOCAL_ROOT)

import numpy as np
import mne
import tensorpac
from tensorpac import Pac
import matplotlib.pyplot as plt
import pandas as pd
from scipy.signal import hilbert
import seaborn as sns

from membandit.behavior import load_behavior_tables, preprocess_learn_df, preprocess_combined_df

# Statistical functions for optional cluster tests
from mne.stats import (
    permutation_cluster_test,
    permutation_cluster_1samp_test,
    ttest_ind_no_p,
    ttest_1samp_no_p,
)

warnings.filterwarnings('ignore')

print('tensorpac.__version__ =', getattr(tensorpac, '__version__', 'unknown'))
print('tensorpac.__file__ =', tensorpac.__file__)
assert '/Users/salmanqasim/Documents/GitRepos/tensorpac/tensorpac/' in tensorpac.__file__.replace('\\', '/'), (
    'Notebook is not using local patched tensorpac. Restart kernel and run this cell first.'
)




## Runtime Hygiene

This notebook requires the local patched `tensorpac` checkout. After opening the notebook, **restart the kernel** and run the import cell plus the environment sanity-check cell before running PAC cells.


In [ ]:
"""
Environment sanity check: ensure patched Pac.filterfit is active
"""
import inspect
import tensorpac
from tensorpac import Pac

src = inspect.getsource(Pac.filterfit)
strict_shape_guard = 'x_pha.shape == x_amp.shape' in src
print('tensorpac runtime path:', tensorpac.__file__)
print('strict shape guard present:', strict_shape_guard)
assert not strict_shape_guard, (
    'Unpatched tensorpac detected (strict shape assertion still present). '
    'Restart kernel and rerun the import cell forcing local tensorpac.'
)
print('Environment check passed: patched multivariate filterfit is active.')



In [ ]:
"""
Add custom analysis utilities to path and import
"""
import sys
sys.path.append('/Users/salmanqasim/Documents/GitRepos/LFPAnalysis')
from LFPAnalysis import oscillation_utils



In [ ]:
"""
Data loading and subject selection
"""
base_dir = '/Volumes/T7 Shield'
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion == 1) & (demo_data.Minerva == 1)].copy()
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]

subj_ids = subj_df.MSSMCode.astype(str).values
print(f'Loaded {len(subj_ids)} subjects for analysis')



In [ ]:
"""
Load and preprocess behavior tables via mv_granger_utils
"""
learn_df_raw, combined_df_raw = load_behavior_tables(load_dir)
learn_df = preprocess_learn_df(learn_df_raw)
combined_df = preprocess_combined_df(combined_df_raw)

print(f'learn_df shape: {learn_df.shape}')
print(f'combined_df shape: {combined_df.shape}')
print('combined required columns present:', {'participant', 'condition', 'trials_gamble', 'hits'}.issubset(combined_df.columns))



In [ ]:
"""
Analysis configuration
"""
evs = {
    'gamble_start': [-1.0, 0.0],
    'feedback_start': [0.0, 1.5],
    'baseline_start': [0.0, 0.75],
    'cue_on': [0.0, 1.0],
}
analysis_evs = ['cue_on']
baseline_ev = 'baseline_start'
day = 'Day1'

for ev in analysis_evs:
    if ev not in evs:
        raise ValueError(f'Analysis event "{ev}" is not defined in evs')

print('Analysis events:', analysis_evs)
print('Day:', day)



In [ ]:
"""
Frequency band configuration
"""
phase_frequency = 'theta'
f_pha_start, f_pha_end = 4, 8
f_pha_nb = 5
f_pha_mid = np.linspace(f_pha_start, f_pha_end, f_pha_nb)
f_pha = np.c_[f_pha_mid - f_pha_mid / 4.0, f_pha_mid + f_pha_mid / 4.0]

amp_frequency = 'hfa'
f_amp_start, f_amp_end = 70, 200
f_amp_nb = 5
f_amp_mid = np.linspace(f_amp_start, f_amp_end, f_amp_nb)
f_amp = np.c_[f_amp_mid - f_amp_mid / 8.0, f_amp_mid + f_amp_mid / 8.0]

print(f'Phase band: {phase_frequency} ({f_pha_start}-{f_pha_end} Hz, {f_pha_nb} bins)')
print(f'Amplitude band: {amp_frequency} ({f_amp_start}-{f_amp_end} Hz, {f_amp_nb} bins)')



## Multivariate Method Note

This notebook computes **ROI-hemi multivariate group-to-group PAC** using Gaussian-Copula PAC (`idpac[0] == 6`).
Each estimate uses all phase channels in the selected phase ROI-hemi and all amplitude channels in the selected amplitude ROI-hemi, rather than channel pairs.


In [ ]:
"""
Output directory configuration
"""
save_dir_data = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/pac'
save_dir_plots = '/Users/salmanqasim/Documents/Projects/MemBanditsEMU/Plots/MemoryBandit-CrossRegionMV-GCPAC'
os.makedirs(save_dir_data, exist_ok=True)
os.makedirs(save_dir_plots, exist_ok=True)

print('Data save dir:', save_dir_data)
print('Plot save dir:', save_dir_plots)



In [ ]:
"""
Helper functions for multivariate GC-PAC computation, trial-level tables, and visualization
"""

def compute_pac_for_group(pac_obj, ph_sig, amp_sig, sfreq, edges, n_perm=250, random_state=0):
    """Compute multivariate PAC for one ROI-hemi channel group.

    If n_perm <= 0, return raw PAC in place of z-scored PAC so downstream
    notebook code can continue to use the same variable names.
    """
    epoch_pac = pac_obj.filterfit(
        sfreq,
        ph_sig,
        amp_sig,
        n_perm=n_perm,
        n_jobs=-1,
        edges=edges,
        random_state=random_state,
    )

    if (not isinstance(n_perm, int)) or (n_perm <= 0) or (pac_obj.surrogates is None):
        z_scored_pac = epoch_pac.copy()
    else:
        surr_mean = pac_obj.surrogates.mean(axis=0)
        surr_std = pac_obj.surrogates.std(axis=0)
        with np.errstate(divide="ignore", invalid="ignore"):
            z_scored_pac = (epoch_pac - surr_mean) / surr_std

    return z_scored_pac, epoch_pac

def normalize_picks(picks_payload):
    """Normalize nested pick payloads into flat unique integer picks."""
    flat = []

    def _walk(x):
        if x is None:
            return
        if isinstance(x, (list, tuple, np.ndarray)):
            for y in np.asarray(x, dtype=object).ravel():
                _walk(y)
            return
        try:
            if pd.isna(x):
                return
        except Exception:
            pass
        try:
            flat.append(int(x))
        except Exception:
            return

    _walk(picks_payload)

    out = []
    seen = set()
    for k in flat:
        if k not in seen:
            out.append(k)
            seen.add(k)
    return out

def create_pac_dataframe(
    z_scored_pac,
    subj_id,
    subj_info,
    phase_roi,
    amp_roi,
    event,
    phase_freq,
    amp_freq,
    phase_channels,
    amp_channels,
    hemi,
    learn_df,
    combined_df,
):
    avg_conn = np.nanmean(z_scored_pac, axis=(0, 1))
    z_scored_pac_sig = z_scored_pac.copy()
    z_scored_pac_sig[np.abs(z_scored_pac_sig) <= 1.96] = np.nan
    sum_conn = np.nansum(z_scored_pac_sig, axis=(0, 1))
    phase_channels = [str(ch) for ch in list(phase_channels)]
    amp_channels = [str(ch) for ch in list(amp_channels)]
    pwise_df = pd.DataFrame({
        "participant": subj_id,
        "age": subj_info["Age"],
        "sex": subj_info["Sex"].strip() if isinstance(subj_info["Sex"], str) else subj_info["Sex"],
        "amp_roi": amp_roi,
        "phase_roi": phase_roi,
        "event": event,
        "band0": phase_freq,
        "band1": amp_freq,
        "trials": np.arange(1, len(sum_conn) + 1),
        "label": f"{phase_roi}:{amp_roi}:{hemi}",
        "phase_channels": "|".join(phase_channels),
        "amp_channels": "|".join(amp_channels),
        "n_phase_channels": len(phase_channels),
        "n_amp_channels": len(amp_channels),
        "hemi": hemi,
        "avg_conn": avg_conn,
        "sum_conn": sum_conn,
    })
    if str(event).strip().lower() == "cue_on":
        cdf = combined_df[combined_df.participant == subj_id].copy()
        if "condition" in cdf.columns:
            cdf = cdf[cdf["condition"] == "Day1"].copy()
        if "trials_mem" in cdf.columns:
            cdf = cdf.rename(columns={"trials_mem": "trials"})
        pwise_df = pwise_df.merge(cdf, on=["participant", "trials"], how="left")
    else:
        subj_learn_df = learn_df[learn_df.participant == subj_id].copy()
        pwise_df = pwise_df.merge(subj_learn_df, on=["participant", "trials"], how="left")
    if "rpe" in pwise_df.columns:
        denom = 2.0 * np.nanstd(pwise_df["rpe"])
        pwise_df["zrpe"] = (pwise_df["rpe"] - np.nanmean(pwise_df["rpe"])) / denom if np.isfinite(denom) and denom > 0 else np.nan
    if "DPRIME" in pwise_df.columns:
        denom = 2.0 * np.nanstd(pwise_df["DPRIME"])
        pwise_df["zpm"] = (pwise_df["DPRIME"] - np.nanmean(pwise_df["DPRIME"])) / denom if np.isfinite(denom) and denom > 0 else np.nan
    return pwise_df

def compute_high_pac_mask(trial_pac_z, conditions, high_pac_quantile=0.9, min_high_trials=5):
    trial_pac_z = np.asarray(trial_pac_z, dtype=float)
    conditions = np.asarray(conditions, dtype=object)
    mask = np.zeros(trial_pac_z.shape[0], dtype=bool)
    for cond in np.unique(conditions):
        if pd.isna(cond) or (isinstance(cond, str) and not str(cond).strip()):
            continue
        cond_mask = (conditions == cond) & np.isfinite(trial_pac_z)
        n_cond = int(cond_mask.sum())
        if n_cond == 0:
            continue
        cond_scores = trial_pac_z[cond_mask]
        threshold = np.nanquantile(cond_scores, high_pac_quantile)
        cond_selected = np.where(cond_mask & (trial_pac_z >= threshold))[0]
        min_take = min(min_high_trials, n_cond)
        if len(cond_selected) < min_take:
            cond_indices = np.where(cond_mask)[0]
            rank_order = np.argsort(cond_scores)[::-1]
            cond_selected = cond_indices[rank_order[:min_take]]
        mask[cond_selected] = True
    return mask

def summarize_by_condition(pwise_df, condition_col="hit_miss", valid_col=None):
    valid_df = pwise_df.copy()
    if isinstance(valid_col, str) and (valid_col in valid_df.columns):
        valid_df = valid_df[valid_df[valid_col] == True].copy()
    if condition_col not in valid_df.columns:
        return {}
    cond_vals = valid_df[condition_col].dropna().unique()
    cond_vals = [c for c in cond_vals if (isinstance(c, str) and c.strip()) or (not isinstance(c, str))]
    out = {}
    means_avg = {}
    means_sum = {}
    for cond in cond_vals:
        sub = valid_df[valid_df[condition_col] == cond]
        means_avg[cond] = float(np.nanmean(sub["avg_conn"])) if len(sub) > 0 else np.nan
        means_sum[cond] = float(np.nanmean(sub["sum_conn"])) if len(sub) > 0 else np.nan
        out[f"mean_avg_conn_{cond}"] = means_avg[cond]
        out[f"mean_sum_conn_{cond}"] = means_sum[cond]
    if len(cond_vals) == 2:
        c1, c2 = cond_vals[0], cond_vals[1]
        out[f"delta_{c1}_minus_{c2}_avg_conn"] = means_avg[c1] - means_avg[c2] if np.isfinite(means_avg[c1]) and np.isfinite(means_avg[c2]) else np.nan
        out[f"delta_{c1}_minus_{c2}_sum_conn"] = means_sum[c1] - means_sum[c2] if np.isfinite(means_sum[c1]) and np.isfinite(means_sum[c2]) else np.nan
    return out

def _collapse_group_trials(x):
    x = np.asarray(x, dtype=float)
    if x.ndim == 3:
        return np.nanmean(x, axis=1)
    if x.ndim == 2:
        return x
    return None

def plot_high_pac_lfp_phase_locking(phase_sig, amp_sig, trial_pac_z, hit_miss_labels, sfreq, times, f_pha_band, f_amp_band, out_file, title_prefix, high_pac_quantile=0.9, min_high_trials=5):
    phase_sig = _collapse_group_trials(phase_sig)
    amp_sig = _collapse_group_trials(amp_sig)
    trial_pac_z = np.asarray(trial_pac_z, dtype=float)
    hit_miss_labels = np.asarray(hit_miss_labels, dtype=object)
    if phase_sig is None or amp_sig is None or phase_sig.shape != amp_sig.shape or phase_sig.shape[0] != len(trial_pac_z):
        return None
    nyquist = sfreq / 2.0
    lf_low = max(float(f_pha_band[0]), 0.5)
    lf_high = min(float(f_pha_band[1]), nyquist - 1.0)
    hf_low = max(float(f_amp_band[0]), 1.0)
    hf_high = min(float(f_amp_band[1]), nyquist - 1.0)
    if lf_high <= lf_low or hf_high <= hf_low:
        return None
    high_mask = compute_high_pac_mask(trial_pac_z=trial_pac_z, conditions=hit_miss_labels, high_pac_quantile=high_pac_quantile, min_high_trials=min_high_trials)
    unique_conds = [c for c in np.unique(hit_miss_labels) if pd.notna(c) and (not isinstance(c, str) or c.strip())]
    cond_indices = {cond: np.where(high_mask & (hit_miss_labels == cond) & np.isfinite(trial_pac_z))[0] for cond in unique_conds}
    if all(len(v) == 0 for v in cond_indices.values()):
        return None
    phase_bins = np.linspace(-np.pi, np.pi, 19)
    phase_centers = (phase_bins[:-1] + phase_bins[1:]) / 2.0
    n_conds = len(unique_conds)
    fig, axes = plt.subplots(n_conds, 3, figsize=(16, 4 * n_conds), sharex="col")
    if n_conds == 1:
        axes = axes[np.newaxis, :]
    for row_idx, cond in enumerate(unique_conds):
        idx = cond_indices[cond]
        ax_lf, ax_hf, ax_phase = axes[row_idx]
        if len(idx) == 0:
            for ax in (ax_lf, ax_hf, ax_phase):
                ax.text(0.5, 0.5, f"No high-PAC {cond} trials", ha="center", va="center")
                ax.set_xticks([]); ax.set_yticks([])
            continue
        lf_data = mne.filter.filter_data(phase_sig[idx], sfreq=sfreq, l_freq=lf_low, h_freq=lf_high, verbose=False)
        hf_data = mne.filter.filter_data(amp_sig[idx], sfreq=sfreq, l_freq=hf_low, h_freq=hf_high, verbose=False)
        hf_env = np.abs(hilbert(hf_data, axis=-1))
        lf_phase = np.angle(hilbert(lf_data, axis=-1))
        mean_lf = np.nanmean(lf_data, axis=0)
        mean_hf_env = np.nanmean(hf_env, axis=0)
        phase_flat = lf_phase.ravel()
        env_flat = hf_env.ravel()
        phase_bin_idx = np.digitize(phase_flat, phase_bins) - 1
        phase_profile = np.asarray([np.nanmean(env_flat[phase_bin_idx == b]) if np.any(phase_bin_idx == b) else np.nan for b in range(len(phase_bins) - 1)], dtype=float)
        cond_colors = ["#2ecc71", "#e74c3c", "#3498db", "#9b59b6", "#f39c12"]
        clr = cond_colors[row_idx % len(cond_colors)]
        ax_lf.plot(times, mean_lf, color=clr, linewidth=1.5)
        ax_lf.set_title(f"{str(cond).upper()} LF ({lf_low:.1f}-{lf_high:.1f} Hz) | n={len(idx)}")
        ax_lf.set_ylabel("LF signal")
        ax_hf.plot(times, mean_hf_env, color=clr, linewidth=1.5)
        ax_hf.set_title(f"{str(cond).upper()} HF envelope ({hf_low:.1f}-{hf_high:.1f} Hz)")
        ax_hf.set_ylabel("HF envelope")
        ax_phase.plot(phase_centers, phase_profile, color=clr, linewidth=1.5)
        ax_phase.set_title(f"{str(cond).upper()} phase-binned HF envelope")
        ax_phase.set_xlabel("LF phase (rad)")
        ax_phase.set_ylabel("Mean HF envelope")
    axes[-1, 0].set_xlabel("Time (s)")
    axes[-1, 1].set_xlabel("Time (s)")
    fig.suptitle(f"{title_prefix} | High z-scored PAC trials", fontsize=12)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(out_file, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return out_file



In [ ]:
"""
Cross-region multivariate GC-PAC main loop: trial-level PAC for every trial; behavioral metadata attached for post-hoc splits.
"""
n_perm = 250
buf_ms = 1000
high_pac_quantile = 0.9
min_high_trials = 5

roi_pairs = [['HPC', 'ACC'],
['HPC', 'HPC'],
['ACC', 'ACC'],
['ACC', 'HPC']]

pac_obj = Pac(idpac=(6, 1, 0), f_pha=f_pha, f_amp=f_amp, dcomplex="wavelet")

all_dfs = []
subjects_to_process = subj_ids

print("\n" + "=" * 70)
print("Starting CROSS-REGION MULTIVARIATE GC-PAC computation")
print("Subjects:", len(subjects_to_process))
print("Events:", analysis_evs)
print("ROI pairs:", roi_pairs)
print("=" * 70 + "\n")

for subj_idx, subj_id in enumerate(subjects_to_process):
    print(f"[{subj_idx + 1}/{len(subjects_to_process)}] Subject: {subj_id}")
    subj_info = subj_df[subj_df.MSSMCode.astype(str) == str(subj_id)].iloc[0]
    elec_df_path = f"{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df"
    if not os.path.exists(elec_df_path):
        print("  Missing electrode table:", elec_df_path)
        continue
    elec_df = pd.read_csv(elec_df_path)
    for event in analysis_evs:
        save_path = f"{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}"
        epochs_file = f"{save_path}/{event}-epo.fif"
        if not os.path.exists(epochs_file):
            print("  Missing epochs file:", epochs_file)
            continue
        print("  Event:", event)
        epochs_reref = mne.read_epochs(epochs_file, preload=True)
        sfreq = float(epochs_reref.info["sfreq"])
        edges = int(buf_ms * (sfreq / 1000.0))
        for phase_roi, amp_roi in roi_pairs:
            print(f"    ROI pair: {phase_roi} -> {amp_roi}")
            seed_target_df = oscillation_utils.make_seed_target_df(elec_df, epochs_reref, phase_roi, amp_roi)
            seed_target_df = seed_target_df.rename(columns={"seed": "phase", "target": "amp"})
            pwise_dfs = []
            for hemi in ["l", "r"]:
                if hemi not in seed_target_df.index.tolist():
                    continue
                phase_group_picks = normalize_picks(seed_target_df["phase"][hemi])
                amp_group_picks = normalize_picks(seed_target_df["amp"][hemi])
                if len(phase_group_picks) == 0 or len(amp_group_picks) == 0:
                    continue
                print(f"      {hemi.upper()} hemisphere group: n_phase={len(phase_group_picks)}, n_amp={len(amp_group_picks)}")
                pac_file = f"{save_dir_data}/{subj_id}_{phase_roi}-{amp_roi}_{event}_{hemi}_mv_gcpac.npy"
                ph_sig = epochs_reref.copy().pick(phase_group_picks).get_data()
                amp_sig = epochs_reref.copy().pick(amp_group_picks).get_data()
                z_scored_pac, _ = compute_pac_for_group(pac_obj=pac_obj, ph_sig=ph_sig, amp_sig=amp_sig, sfreq=sfreq, edges=edges, n_perm=n_perm, random_state=0)
                np.save(pac_file, z_scored_pac)
                trial_pac_z = np.nanmean(z_scored_pac, axis=(0, 1))
                phase_channels = [epochs_reref.ch_names[i] for i in phase_group_picks]
                amp_channels = [epochs_reref.ch_names[i] for i in amp_group_picks]
                pwise_df = create_pac_dataframe(z_scored_pac=z_scored_pac, subj_id=str(subj_id), subj_info=subj_info, phase_roi=phase_roi, amp_roi=amp_roi, event=event, phase_freq=phase_frequency, amp_freq=amp_frequency, phase_channels=phase_channels, amp_channels=amp_channels, hemi=hemi, learn_df=learn_df, combined_df=combined_df)
                pwise_df["trial_pac_z"] = trial_pac_z
                pwise_dfs.append(pwise_df)
            if len(pwise_dfs) > 0:
                all_df_pt = pd.concat(pwise_dfs, ignore_index=True)
                output_file = f"{save_dir_data}/{subj_id}_{phase_roi}-{amp_roi}_mv_gcpac_{event}.csv"
                all_df_pt.to_csv(output_file, index=False)
                all_dfs.append(all_df_pt)
                print("      Saved rows:", len(all_df_pt), "|", output_file)

if len(all_dfs) > 0:
    all_results_df = pd.concat(all_dfs, ignore_index=True)
    combined_output = f"{save_dir_data}/all_subjects_mv_gcpac_combined.csv"
    all_results_df.to_csv(combined_output, index=False)
    print("Saved combined trial-level:", combined_output, "| rows:", len(all_results_df))
else:
    print("No cross-region multivariate GC-PAC results to save.")



In [ ]:
all_results_df.keys()

In [ ]:
all_results_df

In [ ]:
sns.barplot(data=all_results_df[all_results_df.phase_roi=='HPC'].dropna(subset=['rpe_cat']).groupby(['participant', 'amp_roi', 'phase_roi']).mean(numeric_only=True), 
x='amp_roi', y='avg_conn', errorbar='se')

In [ ]:
sns.barplot(data=all_results_df[(all_results_df.phase_roi=='ACC') & (all_results_df.amp_roi=='ACC')], 
x='amp_roi', y='avg_conn', errorbar='se')

In [ ]:
z_scored_pac

In [ ]:
all_results_df

In [ ]:
amp_sig

In [ ]:
pac_obj.comodulogram(z_scored_pac.mean(-1), cmap='viridis')


## Output Checks

In [ ]:
"""
Load and inspect combined trial-level export
"""
trial_file = f"{save_dir_data}/all_subjects_mv_gcpac_combined.csv"

if os.path.exists(trial_file):
    trial_df = pd.read_csv(trial_file)
    print("Trial-level rows:", len(trial_df))
    expected_cols = ["participant", "event", "phase_roi", "amp_roi", "hemi", "trials", "trial_pac_z", "hit_miss", "avg_conn", "sum_conn", "n_phase_channels", "n_amp_channels", "phase_channels", "amp_channels"]
    print("Required columns present:", {c: (c in trial_df.columns) for c in expected_cols})
    if "hit_miss" in trial_df.columns:
        print("Unique hit_miss labels:", sorted(trial_df["hit_miss"].dropna().astype(str).unique().tolist()))
    if "rpe_cat" in trial_df.columns:
        print("Unique rpe_cat labels:", sorted(trial_df["rpe_cat"].dropna().astype(str).unique().tolist()))
else:
    print("Missing trial export:", trial_file)



## Post-hoc trial-level splits

Apply any trial-level split (e.g. `hit_miss`, `rpe_cat`) to the combined trial-level PAC DataFrame. Use `apply_trial_split` for group means and `compute_high_pac_mask` with any condition column for high-PAC subsets.

In [ ]:
def apply_trial_split(trial_df, condition_col, value_col='trial_pac_z', valid_col=None):
    """Group by condition_col and return summary stats (mean, se, n) for value_col. Optionally restrict to valid_col==True."""
    df = trial_df.copy()
    if isinstance(valid_col, str) and (valid_col in df.columns):
        df = df[df[valid_col] == True]
    if condition_col not in df.columns or value_col not in df.columns:
        return pd.DataFrame()
    g = df.groupby(condition_col, dropna=False)[value_col].agg(['mean', 'sem', 'count'])
    g.columns = [f'{value_col}_mean', f'{value_col}_se', 'n']
    return g.reset_index()


In [ ]:
# # Use trial-level DataFrame from memory or from disk (trial_df / all_results_df)
# pac_df = trial_df if 'trial_df' in dir() and isinstance(trial_df, pd.DataFrame) and len(trial_df) > 0 else (
#     all_results_df if 'all_results_df' in dir() and isinstance(all_results_df, pd.DataFrame) else None
# )
# if pac_df is None or len(pac_df) == 0:
#     print('Run the main loop or load the combined CSV first (trial_df or all_results_df).')
# else:
#     # Example: summary by hit_miss
#     print('Summary by hit_miss:')
#     display(apply_trial_split(pac_df, 'hit_miss', value_col='trial_pac_z'))
#     # Example: summary by rpe_cat if present
#     if 'rpe_cat' in pac_df.columns:
#         print('Summary by rpe_cat:')
#         display(apply_trial_split(pac_df, 'rpe_cat', value_col='trial_pac_z'))
#     # Per-pair summarize_by_condition (e.g. for one subject/ROI pair)
#     example = pac_df.dropna(subset=['hit_miss']).head(80)
#     if len(example) > 0:
#         s = summarize_by_condition(example, condition_col='hit_miss')
#         print('Example summarize_by_condition (hit_miss):', s)


## Figure S6B

In [ ]:
phase_roi = 'HPC'
amp_roi = 'HPC'
pac_df = trial_df[trial_df.event == 'cue_on'].dropna(subset=['rpe_cat']).copy()
pac_df = pac_df[(pac_df.phase_roi == phase_roi) & (pac_df.amp_roi == amp_roi)]

if 'rpe_cat' in pac_df.columns:
    import os
    valid_rpe = pac_df.dropna(subset=['rpe_cat', 'avg_conn']).copy()
    if len(valid_rpe) > 0:
        _rpe_map = {'negative': 'neg', 'neg': 'neg', 'neutral': 'neutral', 'positive': 'pos', 'pos': 'pos'}
        valid_rpe['rpe_cat'] = valid_rpe['rpe_cat'].astype(str).str.strip().str.lower().map(_rpe_map)
        valid_rpe = valid_rpe[valid_rpe['rpe_cat'].isin(['neg', 'neutral', 'pos'])].copy()
        if len(valid_rpe) == 0:
            print('Bar+strip: no rows after filtering to neg/neutral/pos.')
        else:
            if 'phase_roi' in valid_rpe.columns and 'amp_roi' in valid_rpe.columns:
                valid_rpe['pair_label'] = (
                    valid_rpe['phase_roi'].astype(str) + '→' + valid_rpe['amp_roi'].astype(str)
                )
            else:
                valid_rpe['pair_label'] = 'conn'

            valid_rpe['avg_conn_z'] = valid_rpe.groupby(['participant', 'pair_label', 'event'])['avg_conn'].transform(
                lambda x: (x - x.mean()) / (x.std(ddof=0) if x.std(ddof=0) > 0 else np.nan)
            )
            plot_df = valid_rpe.copy()
            if not plot_df['avg_conn_z'].notna().any():
                plot_df['avg_conn_z'] = plot_df['avg_conn']

            from matplotlib.lines import Line2D

            cats = ['neg', 'neutral', 'pos']
            class_labels = {'neg': 'negative', 'neutral': 'neutral', 'pos': 'positive'}
            colors = {'neg': '#c0392b', 'neutral': '#7f8c8d', 'pos': '#27ae60'}

            subj_means = (
                plot_df.groupby(['participant', 'rpe_cat'])['avg_conn_z']
                .mean()
                .reset_index()
            )
            bar_df = subj_means.groupby('rpe_cat')['avg_conn_z'].agg(['mean', 'sem']).reindex(cats).reset_index()
            x = np.arange(len(cats))

            fig, ax = plt.subplots(1, 1, figsize=(6.2, 5.6))
            ax.bar(
                x,
                bar_df['mean'].to_numpy(),
                yerr=bar_df['sem'].to_numpy(),
                width=0.62,
                color=[colors[c] for c in cats],
                alpha=0.65,
                edgecolor='none',
                capsize=0,
                zorder=2,
            )

            sns.stripplot(
                data=subj_means,
                x='rpe_cat',
                y='avg_conn_z',
                hue='rpe_cat',
                order=cats,
                hue_order=cats,
                palette=colors,
                dodge=False,
                jitter=0.18,
                alpha=0.55,
                edgecolor='black',
                linewidth=0.8,
                size=4.3,
                ax=ax,
                zorder=4,
            )
            leg = ax.get_legend()
            if leg is not None:
                leg.remove()

            legend_handles = [Line2D([0], [0], color=colors[c], lw=5) for c in cats]
            ax.legend(
                legend_handles,
                [class_labels[c] for c in cats],
                title='RPE',
                frameon=False,
                fontsize=16,
                title_fontsize=24,
                loc='upper right',
                bbox_to_anchor=(1.04, 1.22),
            )

            ax.axhline(0, color='black', linewidth=3.4, zorder=3)
            ax.set_xticks(x)
            ax.set_xticklabels([class_labels[c] for c in cats], fontsize=22)
            ax.set_xlabel('RPE (class)', fontsize=34, labelpad=12)
            ax.set_ylabel('multivariate PAC (z)', fontsize=34, labelpad=10)
            ax.tick_params(axis='x', width=3.2, length=13, pad=9)
            ax.tick_params(axis='y', labelsize=26, width=3.2, length=7, pad=8)
            ax.set_ylim(-0.43, 0.49)
            ax.set_yticks([-0.4, 0.0, 0.4])

            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['left'].set_linewidth(3.2)
            ax.spines['bottom'].set_linewidth(3.2)

            plt.tight_layout()
            export_plot_dir = '/Users/salmanqasim/Documents/Projects/Sinai/MemBanditsEMU/Plots'
            os.makedirs(export_plot_dir, exist_ok=True)
            out_pdf = os.path.join(export_plot_dir, f'{phase_roi}→{amp_roi}_condition_pac.pdf')
            fig.savefig(out_pdf, dpi=300, bbox_inches='tight', facecolor='white')
            print('Saved:', out_pdf)
            plt.show()



In [ ]:
from scipy.stats import ttest_rel
ttest_rel(subj_means[subj_means.rpe_cat=='neutral']['avg_conn_z'], 
subj_means[subj_means.rpe_cat=='neg']['avg_conn_z'])

In [ ]:
# # Post-hoc high-PAC mask with any condition column (e.g. hit_miss or rpe_cat)
# # Example: add is_high_pac for hit_miss split
# if pac_df is not None and len(pac_df) > 0 and 'trial_pac_z' in pac_df.columns:
#     import seaborn as sns
#     # Bar plot by hit_miss
#     fig, ax = plt.subplots(1, 1, figsize=(6, 4))
#     valid = pac_df.dropna(subset=['hit_miss', 'trial_pac_z'])
#     if len(valid) > 0:
#         sns.barplot(data=valid, x='hit_miss', y='trial_pac_z', errorbar='se', ax=ax)
#         ax.set_title('trial_pac_z by hit_miss (post-hoc)')
#         plt.tight_layout()
#         plt.show()
#     # Bar plot by rpe_cat if present
#     if 'rpe_cat' in pac_df.columns:
#         valid_rpe = pac_df.dropna(subset=['rpe_cat', 'trial_pac_z'])
#         if len(valid_rpe) > 0:
#             fig, ax = plt.subplots(1, 1, figsize=(6, 4))
#             sns.barplot(data=valid_rpe, x='rpe_cat', y='trial_pac_z', errorbar='se', ax=ax)
#             ax.set_title('trial_pac_z by rpe_cat (post-hoc)')
#             plt.tight_layout()
#             plt.show()


In [ ]:
model_ = smf.mixedlm(formula="avg_conn ~ hits*C(rpe_cat)", 
data=pac_df, 
groups='participant').fit()
print(model_.summary())

In [ ]:
sns.pointplot(data=pac_df, x='rpe_cat', y='avg_conn', hue='hits', order=['neg', 'neutral', 'pos'], errorbar='se')


## RPE Category Stats: pos vs neutral

Within-subject+hemi comparison of `rpe_cat == pos` vs `rpe_cat == neutral` using trial-summary tests and cluster permutation tests on PAC heatmaps.


In [ ]:
"""
Compute within-subject+hemi pos vs neutral PAC statistics
"""
import os
import numpy as np
import pandas as pd
from scipy import stats as sp_stats
from mne.stats import permutation_cluster_test

def _canon_rpe_cat(v):
    if pd.isna(v):
        return np.nan
    s = str(v).strip().lower()
    mapping = {
        "positive": "pos", "pos": "pos",
        "neutral": "neutral",
        "negative": "neg", "neg": "neg",
    }
    return mapping.get(s, s)

def _unit_pac_file(row, save_dir):
    subj = str(row["participant"])
    ev = str(row["event"])
    phase_roi = str(row["phase_roi"])
    amp_roi = str(row["amp_roi"])
    hemi = str(row["hemi"])
    return f"{save_dir}/{subj}_{phase_roi}-{amp_roi}_{ev}_{hemi}_mv_gcpac.npy"

# Source dataframe
if "trial_df" in locals() and isinstance(trial_df, pd.DataFrame) and len(trial_df) > 0:
    pac_df = trial_df.copy()
elif "all_results_df" in locals() and isinstance(all_results_df, pd.DataFrame) and len(all_results_df) > 0:
    pac_df = all_results_df.copy()
else:
    trial_file = f"{save_dir_data}/all_subjects_mv_gcpac_combined.csv"
    if not os.path.exists(trial_file):
        raise FileNotFoundError(f"Missing trial export: {trial_file}")
    pac_df = pd.read_csv(trial_file)

required_cols = ["participant", "event", "phase_roi", "amp_roi", "hemi", "trials", "trial_pac_z", "rpe_cat"]
missing = [c for c in required_cols if c not in pac_df.columns]
if len(missing) > 0:
    raise ValueError(f"Missing required columns for RPE stats: {missing}")

df = pac_df.copy()
df = df.dropna(subset=['rpe_cat']).copy()
df["rpe_cat_norm"] = df["rpe_cat"].map(_canon_rpe_cat)
df["sum_conn"] = pd.to_numeric(df["sum_conn"], errors="coerce")
df["trials"] = pd.to_numeric(df["trials"], errors="coerce")
df = df[np.isfinite(df["sum_conn"]) & df["rpe_cat_norm"].isin(["pos", "neutral"])].copy()

unit_cols = ["participant", "event", "phase_roi", "amp_roi", "hemi"]
units = df[unit_cols].drop_duplicates().reset_index(drop=True)
print("Units to test:", len(units))

n_perm_cluster = 500
alpha = 0.05
rows = []

for _, u in units.iterrows():
    u_mask = np.logical_and.reduce([df[c].astype(str).values == str(u[c]) for c in unit_cols])
    u_df = df.loc[u_mask].copy()
    if len(u_df) == 0:
        continue

    pos_df = u_df[u_df["rpe_cat_norm"] == "pos"].copy()
    neu_df = u_df[u_df["rpe_cat_norm"] == "neutral"].copy()
    n_pos = int(len(pos_df))
    n_neutral = int(len(neu_df))

    # Trial-summary test
    common_trials = sorted(set(pos_df["trials"].dropna().astype(int).tolist()) & set(neu_df["trials"].dropna().astype(int).tolist()))
    t_stat = np.nan
    p_t = np.nan
    t_test_type = "insufficient"
    delta_mean = np.nan

    if len(common_trials) >= 2:
        pos_vec = pos_df.set_index(pos_df["trials"].astype(int))["trial_pac_z"].reindex(common_trials).to_numpy(dtype=float)
        neu_vec = neu_df.set_index(neu_df["trials"].astype(int))["trial_pac_z"].reindex(common_trials).to_numpy(dtype=float)
        ok = np.isfinite(pos_vec) & np.isfinite(neu_vec)
        if ok.sum() >= 2:
            t_stat, p_t = sp_stats.ttest_rel(pos_vec[ok], neu_vec[ok], nan_policy="omit")
            t_test_type = "paired_t"
            delta_mean = float(np.nanmean(pos_vec[ok] - neu_vec[ok]))
    elif (n_pos >= 2) and (n_neutral >= 2):
        pos_vec = pos_df["sum_conn"].to_numpy(dtype=float)
        neu_vec = neu_df["sum_conn"].to_numpy(dtype=float)
        t_stat, p_t = sp_stats.ttest_ind(pos_vec, neu_vec, equal_var=False, nan_policy="omit")
        t_test_type = "ind_t_fallback"
        delta_mean = float(np.nanmean(pos_vec) - np.nanmean(neu_vec))

    # Cluster permutation on heatmaps
    min_cluster_p = np.nan
    n_sig_clusters = 0
    cluster_ok = False

    pac_file = _unit_pac_file(u, save_dir_data)
    if os.path.exists(pac_file) and (n_pos >= 2) and (n_neutral >= 2):
        z = np.load(pac_file)
        if z.ndim == 3:
            # expected: (n_amp, n_pha, n_trials)
            z_trials = np.moveaxis(z, -1, 0)
            trial_labels = u_df[["trials", "rpe_cat_norm"]].dropna().copy()
            trial_labels["trials"] = trial_labels["trials"].astype(int)
            trial_labels = trial_labels.drop_duplicates(subset=["trials"], keep="first")

            pos_idx = [int(t - 1) for t in trial_labels.loc[trial_labels["rpe_cat_norm"] == "pos", "trials"].tolist() if 1 <= int(t) <= z.shape[-1]]
            neu_idx = [int(t - 1) for t in trial_labels.loc[trial_labels["rpe_cat_norm"] == "neutral", "trials"].tolist() if 1 <= int(t) <= z.shape[-1]]

            if (len(pos_idx) >= 2) and (len(neu_idx) >= 2):
                X_pos = z_trials[pos_idx, :, :]
                X_neu = z_trials[neu_idx, :, :]
                try:
                    T_obs, clusters, cluster_pv, _ = permutation_cluster_test(
                        [X_pos, X_neu],
                        n_permutations=n_perm_cluster,
                        tail=0,
                        out_type="mask",
                        verbose=False,
                    )
                    if len(cluster_pv) > 0:
                        min_cluster_p = float(np.nanmin(cluster_pv))
                        n_sig_clusters = int(np.sum(np.asarray(cluster_pv) < alpha))
                    cluster_ok = True
                except Exception as e:
                    cluster_ok = False

    rows.append({
        **{c: u[c] for c in unit_cols},
        "n_pos": n_pos,
        "n_neutral": n_neutral,
        "t_test_type": t_test_type,
        "t_stat": t_stat,
        "p_t": p_t,
        "delta_mean_pos_minus_neutral": delta_mean,
        "cluster_ok": cluster_ok,
        "min_cluster_p": min_cluster_p,
        "n_sig_clusters": n_sig_clusters,
        "ttest_sig": bool(np.isfinite(p_t) and (p_t < alpha)),
        "cluster_sig": bool(np.isfinite(min_cluster_p) and (min_cluster_p < alpha)),
        "pac_file": pac_file,
    })

rpe_stats_df = pd.DataFrame(rows)
if len(rpe_stats_df) == 0:
    print("No units produced pos vs neutral stats.")
else:
    rpe_stats_df["sig_either"] = rpe_stats_df["ttest_sig"] | rpe_stats_df["cluster_sig"]
    out_csv = f"{save_dir_data}/crossregion_mv_gcpac_rpe_pos_vs_neutral_stats.csv"
    rpe_stats_df.to_csv(out_csv, index=False)
    print("Saved stats:", out_csv)
    print("Units tested:", len(rpe_stats_df))
    print("Significant via t-test:", int(rpe_stats_df["ttest_sig"].sum()))
    print("Significant via cluster:", int(rpe_stats_df["cluster_sig"].sum()))
    print("Significant via either:", int(rpe_stats_df["sig_either"].sum()))
    display(rpe_stats_df.sort_values(["sig_either", "p_t", "min_cluster_p"], ascending=[False, True, True]).head(20))



## Figure S6A

## Plot Significant `pos` vs `neutral` Units (Either Test)

Plot condition-wise mean comodulograms for units significant by either the trial-summary test or the cluster permutation test.


In [ ]:
"""
Plot pos/neutral mean comodulograms for units significant by either test
"""
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

export_plot_dir = "/Users/salmanqasim/Documents/Projects/Sinai/MemBanditsEMU/Plots"
os.makedirs(export_plot_dir, exist_ok=True)

if "rpe_stats_df" not in locals() or not isinstance(rpe_stats_df, pd.DataFrame) or len(rpe_stats_df) == 0:
    stats_file = f"{save_dir_data}/crossregion_mv_gcpac_rpe_pos_vs_neutral_stats.csv"
    if not os.path.exists(stats_file):
        raise FileNotFoundError(f"Missing stats file: {stats_file}")
    rpe_stats_df = pd.read_csv(stats_file)
    if "sig_either" not in rpe_stats_df.columns:
        rpe_stats_df["sig_either"] = rpe_stats_df["ttest_sig"].astype(bool) | rpe_stats_df["cluster_sig"].astype(bool)

if "trial_df" in locals() and isinstance(trial_df, pd.DataFrame) and len(trial_df) > 0:
    pac_df_plot = trial_df.copy()
elif "all_results_df" in locals() and isinstance(all_results_df, pd.DataFrame) and len(all_results_df) > 0:
    pac_df_plot = all_results_df.copy()
else:
    trial_file = f"{save_dir_data}/all_subjects_mv_gcpac_combined.csv"
    if not os.path.exists(trial_file):
        raise FileNotFoundError(f"Missing trial export: {trial_file}")
    pac_df_plot = pd.read_csv(trial_file)

def _canon_rpe_cat_plot(v):
    if pd.isna(v):
        return np.nan
    s = str(v).strip().lower()
    return {"positive": "pos", "pos": "pos", "neutral": "neutral", "negative": "neg", "neg": "neg"}.get(s, s)

pac_df_plot = pac_df_plot.dropna(subset=['rpe_cat']).copy()
pac_df_plot["rpe_cat_norm"] = pac_df_plot["rpe_cat"].map(_canon_rpe_cat_plot)
pac_df_plot["trials"] = pd.to_numeric(pac_df_plot["trials"], errors="coerce")

if "pac_obj" in locals():
    pac_plot_obj = pac_obj
else:
    pac_plot_obj = Pac(idpac=(6, 0, 0), f_pha=f_pha, f_amp=f_amp, dcomplex="wavelet")

sig_units = rpe_stats_df[rpe_stats_df["sig_either"] == True].copy()
if len(sig_units) == 0:
    print("No units significant by either test.")
else:
    sig_units = sig_units.sort_values(["p_t", "min_cluster_p"], ascending=[True, True])
    max_plots = min(12, len(sig_units))
    print(f"Plotting {max_plots} significant units")
    unit_cols = ["participant", "event", "phase_roi", "amp_roi", "hemi"]

    for _, u in sig_units.head(max_plots).iterrows():
        pac_file = str(u.get("pac_file", ""))
        if (not pac_file) or (not os.path.exists(pac_file)):
            pac_file = f"{save_dir_data}/{u['participant']}_{u['phase_roi']}-{u['amp_roi']}_{u['event']}_{u['hemi']}_mv_gcpac.npy"
        if not os.path.exists(pac_file):
            print("Missing PAC tensor:", pac_file)
            continue

        z = np.load(pac_file)
        if z.ndim != 3:
            print("Unexpected PAC tensor shape:", z.shape, pac_file)
            continue

        mask = np.logical_and.reduce([pac_df_plot[c].astype(str).values == str(u[c]) for c in unit_cols])
        u_df = pac_df_plot.loc[mask, ["trials", "rpe_cat_norm"]].dropna().copy()
        if len(u_df) == 0:
            continue
        u_df["trials"] = u_df["trials"].astype(int)
        u_df = u_df.drop_duplicates(subset=["trials"], keep="first")

        pos_idx = [int(t - 1) for t in u_df.loc[u_df["rpe_cat_norm"] == "pos", "trials"].tolist() if 1 <= int(t) <= z.shape[-1]]
        neu_idx = [int(t - 1) for t in u_df.loc[u_df["rpe_cat_norm"] == "neutral", "trials"].tolist() if 1 <= int(t) <= z.shape[-1]]
        if (len(pos_idx) == 0) or (len(neu_idx) == 0):
            continue

        z_pos = np.nanmean(z[..., pos_idx], axis=-1)
        z_neu = np.nanmean(z[..., neu_idx], axis=-1)
        z_diff = z_pos - z_neu

        plt.figure(figsize=(14, 4))
        title_head = f"{u['participant']} {u['event']} {u['phase_roi']}->{u['amp_roi']} hemi={u['hemi']}"

        plt.subplot(1, 3, 1)
        pac_plot_obj.comodulogram(z_pos, cmap="viridis", title=f"{title_head}\npos", vmin=-1, vmax=1)

        plt.subplot(1, 3, 2)
        pac_plot_obj.comodulogram(z_neu, cmap="viridis", title="neutral", vmin=-1, vmax=1)
        plt.ylabel("")

        plt.subplot(1, 3, 3)
        pac_plot_obj.comodulogram(z_diff, cmap="viridis", title="pos - neutral")
        plt.ylabel("")

        plt.tight_layout()
        out_png = os.path.join(
            export_plot_dir,
            (
                f"{u['participant']}_{u['phase_roi']}-{u['amp_roi']}_{u['event']}_{u['hemi']}_pos_neutral_comodulo.pdf"
            ).replace("/", "-"),
        )
        plt.savefig(out_png, dpi=300, bbox_inches="tight")
        print(f"Saved: {out_png}")
        plt.show()

